In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

## Data Check

In [31]:
# --- VQNiche: experiment preflight (summary + sanity checks) ---

from pathlib import Path
import pickle, glob, os, sys
import numpy as np
import torch
import scanpy as sc

# Point to your src (already used in previous notebooks)
SRC = "/nfs/team361/mv11/vqniche/src"
if SRC not in sys.path: sys.path.insert(0, SRC)

from vqniche.dataset.in_memory_dataset_blob import InMemoryDatasetBlob
from vqniche.dataset.transforms import SetExperimentDataKeys

# ---- Paths (as established earlier) ----
DATA_DIR      = Path("/nfs/team361/mv11/DATASETS")
DATASET_NAME  = "merfish_mouse_cortex_from_luna_csv"
SILVER_DIR    = DATA_DIR / "silver" / DATASET_NAME
GOLD_DIR      = DATA_DIR / "gold" / "in-memory-PyG-dataset-blob" / DATASET_NAME

assert SILVER_DIR.exists(), f"Missing silver dir: {SILVER_DIR}"
assert GOLD_DIR.exists(),   f"Missing gold dir:   {GOLD_DIR}"

blob_pt = GOLD_DIR / "dataset_blob.pt"
lab_pkl = GOLD_DIR / "label_categories.pkl"
gp_pkl  = GOLD_DIR / "gene_panel.pkl"

for p in [blob_pt, lab_pkl, gp_pkl]:
    if not p.exists():
        raise FileNotFoundError(
            f"[gold sanity] Expected {p.name} in {GOLD_DIR} but not found.\n"
            "Re-run 00_silver_to_gold_merfish.ipynb (overwrite=True) to rebuild."
        )

# ---- Print quick summary of what we’re about to use ----
print("=== VQNiche – experiment preflight ===")
print("CUDA available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
print("Silver dir:", SILVER_DIR)
print("Gold dir:  ", GOLD_DIR)

with open(lab_pkl, "rb") as f:
    label_categories = pickle.load(f)
labels = label_categories.get("cell_types", [])
print(f"Labels ({len(labels)}):", labels)

with open(gp_pkl, "rb") as f:
    gene_panel = pickle.load(f)
print("Gene panel size:", getattr(gene_panel, "shape", [None])[0])

# ---- Load the gold blob non-destructively ----
graph_kwargs = dict(
    coord_type="generic",
    spatial_key="spatial",
    include_self_loop=False,
    n_neighs_list=[10],
    radius_list=None,
    k={},  # embeddings off for now
)

feature_names = ["cell_gene_counts"]
label_names   = ["cell_types=cell_type"]

blob = InMemoryDatasetBlob(
    name=DATASET_NAME,
    feature_names=feature_names,
    label_names=label_names,
    graph_kwargs=graph_kwargs,
    data_directory_path=DATA_DIR,
    overwrite=False,  # DO NOT rebuild here; just load
)

# discover the edge key that exists in this blob
probe = blob[0]
edge_keys = [k for k in probe.keys() if k.startswith("edge_index_")]
if not edge_keys:
    raise RuntimeError("No edge_index_* keys found; check silver→gold graph construction.")
edge_index_name = edge_keys[0].replace("edge_index_", "")
print("Discovered edge_index_name:", edge_index_name)

# standardize x/y/edge_index for training
set_keys = SetExperimentDataKeys(
    feature_names=["X"],               # raw counts as features (stored internally as X)
    label_name="cell_types",           # one-hot labels
    edge_index_name=edge_index_name,   # e.g., spatial_n_neighs_10
)

dataset = InMemoryDatasetBlob(
    name=DATASET_NAME,
    feature_names=feature_names,
    label_names=label_names,
    graph_kwargs=graph_kwargs,
    data_directory_path=DATA_DIR,
    overwrite=False,
    transform=set_keys,
)

# ---- Dataset stats ----
print("\nBatches (sections):", len(dataset))

def _brief(data):
    E = data.edge_index.shape[1]
    N = data.x.shape[0]
    deg = (2*E)/max(N,1)
    return f"nodes={N}, edges={E}, avg_deg≈{deg:.2f}, feats={data.x.shape[1]}, classes={data.y.shape[1]}"

# print a couple batches as sample
for i in [0, len(dataset)//2, len(dataset)-1]:
    d = dataset[i]
    print(f"[batch {i:02d}] {_brief(d)}")

# ---- Optional: compare silver <> gold counts for a random batch ----
import random, glob
sf = sorted(glob.glob(str(SILVER_DIR / "*.h5ad")))
if sf:
    ad = sc.read_h5ad(random.choice(sf))
    print("\nRandom silver file:", os.path.basename(ad.uns.get("section","?")) if "section" in ad.uns else "(no section in .uns)")
    print("Silver: n_obs×n_vars =", ad.shape, "| spatial:", ad.obsm.get("spatial", np.empty((0,2))).shape)
    print("Gold  : matching batch example:", _brief(dataset[0]))

# ---- Class balance quick look (first batch) ----
b0 = dataset[0]
cls_counts = b0.y.argmax(1).bincount(minlength=b0.y.shape[1]).tolist()
print("\n[first batch] class counts:", cls_counts)

# ---- Persist a tiny context dict for downstream config/logging ----
context = dict(
    dataset_name=DATASET_NAME,
    gold_dir=str(GOLD_DIR),
    num_batches=len(dataset),
    num_features=b0.x.shape[1],
    num_classes=b0.y.shape[1],
    edge_index_name=edge_index_name,
    label_categories=labels,
)
print("\nContext:", context)

# At this point:
# - dataset[i].x / .y / .edge_index are ready for a trainer
# - If following Arpit's suggestion to run VNiche without VQ/GNN/FiLM:
#   * ensure your config disables those blocks and their losses
#   * or temporarily comment them in the encoder and return dummy (h_quantized, indices)
# - If reviving VanillaMLP: point it to this `dataset` and fix the out-of-date bits.


=== VQNiche – experiment preflight ===
CUDA available: True | device count: 1
Silver dir: /nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv
Gold dir:   /nfs/team361/mv11/DATASETS/gold/in-memory-PyG-dataset-blob/merfish_mouse_cortex_from_luna_csv
Labels (23): ['Astro', 'Endo', 'L2/3 IT', 'L4/5 IT', 'L5 ET', 'L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6 IT Car3', 'L6b', 'Lamp5', 'Micro', 'OPC', 'Oligo', 'PVM', 'Peri', 'Pvalb', 'SMC', 'Sncg', 'Sst', 'VLMC', 'Vip']
Gene panel size: 254


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/io/fs.py:229: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "


Discovered edge_index_name: spatial_n_neighs_10

Batches (sections): 64
[batch 00] nodes=5001, edges=56414, avg_deg≈22.56, feats=254, classes=23
[batch 32] nodes=1156, edges=13262, avg_deg≈22.94, feats=254, classes=23
[batch 63] nodes=5643, edges=63708, avg_deg≈22.58, feats=254, classes=23

Random silver file: mouse1_slice212
Silver: n_obs×n_vars = (6458, 254) | spatial: (6458, 2)
Gold  : matching batch example: nodes=5001, edges=56414, avg_deg≈22.56, feats=254, classes=23

[first batch] class counts: [400, 362, 751, 658, 166, 334, 68, 404, 306, 52, 60, 65, 139, 117, 426, 105, 151, 153, 88, 3, 79, 71, 43]

Context: {'dataset_name': 'merfish_mouse_cortex_from_luna_csv', 'gold_dir': '/nfs/team361/mv11/DATASETS/gold/in-memory-PyG-dataset-blob/merfish_mouse_cortex_from_luna_csv', 'num_batches': 64, 'num_features': 254, 'num_classes': 23, 'edge_index_name': 'spatial_n_neighs_10', 'label_categories': ['Astro', 'Endo', 'L2/3 IT', 'L4/5 IT', 'L5 ET', 'L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6 I

In [37]:
# --- Step 2: Build one big PyG graph and initialize the DataModule ---

from pathlib import Path
import sys, torch
from torch_geometric.data import Data
from vqniche.dataset.in_memory_dataset_blob import InMemoryDatasetBlob
from vqniche.dataset.transforms import SetExperimentDataKeys
from vqniche.dataloaders.in_memory_datamodule import InMemoryDataModule

# --- paths (same as preflight) ---
DATA_DIR      = Path("/nfs/team361/mv11/DATASETS")
DATASET_NAME  = "merfish_mouse_cortex_from_luna_csv"

# --- use the repo loader exactly as in preflight ---
graph_kwargs = dict(
    coord_type="generic",
    spatial_key="spatial",
    include_self_loop=False,
    n_neighs_list=[10],
    radius_list=None,
    k={},
)
feature_names = ["cell_gene_counts"]
label_names   = ["cell_types=cell_type"]

set_keys = SetExperimentDataKeys(
    feature_names=["X"],          # use raw counts as features
    label_name="cell_types",      # one-hot labels
    edge_index_name="spatial_n_neighs_10",  # you discovered this in preflight
)

dataset = InMemoryDatasetBlob(
    name=DATASET_NAME,
    feature_names=feature_names,
    label_names=label_names,
    graph_kwargs=graph_kwargs,
    data_directory_path=DATA_DIR,
    overwrite=False,
    transform=set_keys,
)

print(f"[info] sections: {len(dataset)}")
print(f"[info] first section feats: {dataset[0].x.shape[1]} (should be 254)")

# --- helper: concatenate per-section graphs into ONE big graph ---
def concat_sections(data_list):
    xs, ys, eis, xys = [], [], [], []
    tr_masks, va_masks, te_masks = [], [], []
    batch_ids = []
    offset = 0
    for b, d in enumerate(data_list):
        # ensure CPU tensors
        x   = d.x.cpu()
        y   = d.y.cpu()
        ei  = d.edge_index.cpu() + offset
        xy  = d.xy_coordinates.cpu()
        tm  = d.train_mask.cpu()
        vm  = d.val_mask.cpu()
        sm  = d.test_mask.cpu()
        N   = x.size(0)

        xs.append(x); ys.append(y); eis.append(ei); xys.append(xy)
        tr_masks.append(tm); va_masks.append(vm); te_masks.append(sm)
        # keep track of original section (optional but useful)
        batch_ids.append(torch.full((N,), b, dtype=torch.long))
        offset += N

    big = Data(
        x=torch.cat(xs, 0),
        y=torch.cat(ys, 0),
        edge_index=torch.cat(eis, 1),
        xy_coordinates=torch.cat(xys, 0),
        train_mask=torch.cat(tr_masks, 0),
        val_mask=torch.cat(va_masks, 0),
        test_mask=torch.cat(te_masks, 0),
        adata_batch_ids=torch.cat(batch_ids, 0),
    )
    return big

# materialize all sections and concat
sections = [dataset[i] for i in range(len(dataset))]
big = concat_sections(sections)

print(f"[big] nodes={big.x.size(0)}, edges={big.edge_index.size(1)}, feats={big.x.size(1)}")
assert big.x.size(1) == 254, "Feature dim must be 254 — check silver→gold mapping if this fails."

# --- Initialize the repo DataModule (NeighborLoader + NeighborSampler) ---
dm = InMemoryDataModule(
    data=big,
    loader_name="NeighborLoader",
    loader_params={"batch_size": 1024},
    sampler_name="NeighborSampler",
    sampler_params={"num_neighbors": [5, 5]},
    sample_neighbors_for_inference=False,   # no neighbor sampling in val/test/predict
)

# quick sanity peeks
def _peek(dl, name):
    it = iter(dl)
    try:
        b = next(it)
        print(f"[{name}] batch.x={tuple(b.x.shape)}, edge_index={tuple(b.edge_index.shape)}")
        assert b.x.shape[1] == 254
    except StopIteration:
        print(f"[{name}] empty split")

train_dl = dm.train_dataloader(); _peek(train_dl, "train")
val_dl   = dm.val_dataloader();   _peek(val_dl, "val")
test_dl  = dm.test_dataloader();  _peek(test_dl, "test")
pred_dl  = dm.predict_dataloader(); _peek(pred_dl, "predict")


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/io/fs.py:233: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case.
  warnings.warn(warn_msg)


[info] sections: 64
[info] first section feats: 254 (should be 254)


AttributeError: 'GlobalStorage' object has no attribute 'train_mask'

In [6]:
!source /nfs/team361/mv11/.venvs/LUNA/bin/activate


In [28]:
# --- Environment sanity ---
import sys, os, yaml, inspect
print("Python:", sys.executable)  # expect: /nfs/team361/mv11/.venvs/LUNA/bin/python

# If you still need this wheel pinned:
!{sys.executable} -m pip install -q --no-cache-dir scikit-misc==0.3.1

# --- Paths ---
REPO_SRC = "/nfs/team361/mv11/vqniche/src"
CFG_PATH = "/nfs/team361/mv11/vqniche-reproducibility/config/train_model/merfish_mouse_cortex_from_luna_csv_luna-split_vniche-mlp.yaml"

# Optional: expected gene panel guard-rails
EXPECTED_PANEL_CSV = "/nfs/team361/mv11/DATASETS/luna/MERFISH_mouse_cortex/MERFISH_mouse_cortex_test.csv"
ALIASES_YAML      = "/nfs/team361/mv11/DATASETS/genes/merfish_mouse_cortex_from_luna_csv_aliases.yaml"
EXPECTED_N_GENES  = 254  # set to 249 if you intentionally downselect HVGs

sys.path.insert(0, REPO_SRC)

import pytorch_lightning as pl
import torch

from vqniche.initializers.initialize import (
    initialize_dataset_blob,
    initialize_databatch,
    initialize_datamodule,
    set_model_class,
)

pl.seed_everything(42, workers=True)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())

# Load config once (edited in VS Code as you prefer)
cfg = yaml.safe_load(open(CFG_PATH))


Python: /nfs/team361/mv11/.venvs/LUNA/bin/python


Seed set to 42


CUDA available: True | GPUs: 1


In [29]:
# Build dataset (does NOT rebuild if gold already exists)
dataset_blob = initialize_dataset_blob(cfg)
data_train   = initialize_databatch(cfg, dataset_blob)
datamodule   = initialize_datamodule(cfg, data_train)

print(f"num_features (X): {getattr(data_train, 'num_features', None)}")
print(f"num_classes  (y): {getattr(data_train, 'num_classes', None)}")

# --- Optional: enforce/verify the gene panel & alias mapping ---
# This is a guard so you notice immediately if transforms dropped genes unintentionally.
try:
    import pandas as pd
    panel = pd.read_csv(EXPECTED_PANEL_CSV)
    # The column name holding gene symbols might differ; try common names:
    for col in ("gene", "Gene", "symbol", "Symbol", "gene_name"):
        if col in panel.columns:
            panel_genes = panel[col].astype(str).str.strip().tolist()
            break
    else:
        panel_genes = panel.iloc[:,0].astype(str).str.strip().tolist()

    # Load aliases map if present (useful for "1-Mar" -> "March1", etc.)
    if os.path.exists(ALIASES_YAML):
        alias_map = yaml.safe_load(open(ALIASES_YAML)) or {}
        # normalize both sides to strings
        alias_map = {str(k): str(v) for k, v in alias_map.items()}
    else:
        alias_map = {}

    expected = set(alias_map.get(g, g) for g in panel_genes)
    got_n = int(getattr(data_train, "num_features"))
    if EXPECTED_N_GENES and EXPECTED_N_GENES != got_n:
        print(f"[WARN] Expected ~{EXPECTED_N_GENES} genes, got {got_n}.")
        print("       This typically means transform(s) like SubsetHVG dropped genes.")
        print("       See the config & code patches below to lock a fixed panel and alias mapping.")
except Exception as e:
    print("[Panel check skipped]", e)


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/io/fs.py:229: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "


SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: 

In [30]:
ModelClass = set_model_class(cfg["model"]["model_name"])

def build_model(cfg, data_train):
    """
    Always pass explicit in/out dims to avoid NoneType mistakes in BaseModel.
    If the class accepts a full config (config/cfg/hparams), we also pass that.
    """
    in_ch  = int(getattr(data_train, "num_features"))
    out_ch = int(getattr(data_train, "num_classes"))

    # Prefer explicit constructor for clarity and safety
    ctor_kwargs = dict(cfg["model"])
    ctor_kwargs["in_channels"]  = in_ch
    ctor_kwargs["out_channels"] = out_ch

    sig = inspect.signature(ModelClass.__init__).parameters
    if "config" in sig:
        return ModelClass(config=cfg, **{k:v for k,v in ctor_kwargs.items() if k in sig})
    if "cfg" in sig:
        return ModelClass(cfg=cfg, **{k:v for k,v in ctor_kwargs.items() if k in sig})
    if "hparams" in sig:
        return ModelClass(hparams=cfg, **{k:v for k,v in ctor_kwargs.items() if k in sig})

    # Fallback: only kwargs actually accepted by the model
    safe_kwargs = {k:v for k,v in ctor_kwargs.items() if k in sig}
    return ModelClass(**safe_kwargs)

model = build_model(cfg, data_train)
print("Built model:", type(model).__name__)

# --- Trainer ---
trainer = pl.Trainer(
    accelerator="auto",
    devices="auto",
    max_epochs=int(cfg["trainer"]["max_epochs"]),
    deterministic=True,
    enable_checkpointing=True,
)

print("Fitting…")
trainer.fit(model=model, datamodule=datamodule)

# --- Evaluate on test (respect best.ckpt if present) ---
from pathlib import Path
best_ckpt = None
for p in sorted(Path(str(trainer.logger.log_dir)).rglob("*.ckpt")):  # search current run dir first
    if "last.ckpt" not in p.name:
        best_ckpt = str(p); break

# If you want a different split for testing, clone cfg and change adata_batch_idx etc., then rebuild dm
# Here we test on the same datamodule for simplicity:
Model = set_model_class(cfg["model"]["model_name"])
if best_ckpt:
    best_model = Model.load_from_checkpoint(best_ckpt)
    print("Testing best checkpoint:", best_ckpt)
else:
    best_model = model
    print("Testing current in-memory model")

trainer.test(model=best_model, datamodule=datamodule, ckpt_path=(best_ckpt or None))


Setting the following loss terms as criterion for training:
Loss function: nb_attribute_reconstruction_loss
Loss function: bce_adjacency_reconstruction_loss
Loss function: mse_commit_loss
Loss function: mse_code_loss
Loss function: mask_token_regularization
Setting imputation parameters:
mask_strategy: learnable_parameter
base_mask_ratio: 0.3
final_mask_ratio: 0.3
warmup_epochs: 0
deterministic_masking: False
compute_mask_input_diversity: False
mask_token_eps: 0.0001


TypeError: FiLM.__init__() missing 1 required positional argument: 'condition_dim'

In [10]:
!python -m pip install vector_quantize_pytorch==1.23.2 einops einx frozendict
!python -m pip install scib-metrics

# optional (silences tqdm/ipython widget warning in your logs)
!python -m pip install ipywidgets


  Using cached ipywidgets-8.1.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.14-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.7-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl (216 kB)
Using cached widgetsnbextension-4.0.14-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]━━━━━━━ 2/3 [ipywidgets]]


In [11]:
import os, json, yaml, pickle, glob, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import pytorch_lightning as pl

In [14]:
import sys
sys.path.insert(0, "/nfs/team361/mv11/vqniche/src")

from vqniche.initializers.initialize import (
    initialize_dataset_blob,
    initialize_databatch,
    initialize_datamodule,
    set_model_class,
)

import pytorch_lightning as pl
import torch
pl.seed_everything(42, workers=True)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())


Seed set to 42


CUDA: True | GPUs: 1


Merfish

In [15]:
from pathlib import Path

# data roots (your NFS paths)
DATASET_NAME = "merfish_mouse_cortex_from_luna_csv"
ROOT_DATA_DIR = Path("/nfs/team361/mv11/DATASETS")
SILVER_DIR = ROOT_DATA_DIR / "silver" / DATASET_NAME
GOLD_DIR   = ROOT_DATA_DIR / "gold" / "in-memory-PyG-dataset-blob" / DATASET_NAME

# logs for this run (your home area, not am84’s)
RUN_ROOT = Path("/nfs/team361/mv11/VQNiche/logs") / DATASET_NAME / "vniche_mlp"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# where the training YAML will be written on disk
TRAIN_CFG_DIR = Path("/nfs/team361/mv11/vqniche-reproducibility/config/train_model")
TRAIN_CFG_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CFG = TRAIN_CFG_DIR / f"{DATASET_NAME}_luna-split_vniche-mlp.yaml"
print("Will write training YAML to:", TRAIN_CFG)


Will write training YAML to: /nfs/team361/mv11/vqniche-reproducibility/config/train_model/merfish_mouse_cortex_from_luna_csv_luna-split_vniche-mlp.yaml


In [16]:
import scanpy as sc, json, glob

silver_files = sorted(SILVER_DIR.glob("*.h5ad"))
assert silver_files, f"No silver files in {SILVER_DIR}"

batch_split = {}
for f in silver_files:
    ad = sc.read_h5ad(f)
    bname = ad.uns.get("batch", None)   # e.g., 'batch47'
    assert bname and bname.startswith("batch"), f"No 'batch' in uns for {f}"
    bidx = int(bname.replace("batch",""))
    sflag = str(ad.obs["split"].iloc[0]).lower()  # 'train' or 'test'
    assert sflag in {"train","test"}, f"split must be train/test for {f}"
    batch_split[bidx] = sflag

train_batches = sorted([k for k,v in batch_split.items() if v=="train"])
test_batches  = sorted([k for k,v in batch_split.items() if v=="test"])
print(f"Train sections: {len(train_batches)} | Test sections: {len(test_batches)}")

# persist for auditability
(RUN_ROOT / "luna_section_split.json").write_text(
    json.dumps({"train": train_batches, "test": test_batches}, indent=2)
)
print("Saved:", RUN_ROOT / "luna_section_split.json")


Train sections: 33 | Test sections: 31
Saved: /nfs/team361/mv11/VQNiche/logs/merfish_mouse_cortex_from_luna_csv/vniche_mlp/luna_section_split.json


In [24]:
import yaml

yaml_cfg = {
  "experiment": {"seed": 42, "mode": "standalone"},
  "logging": {
    "root_log_dir": str(RUN_ROOT),
    "log_model": False,
    "offline": True,
    "enabled": True,
  },
  "dataset": {
    "dataset_name": DATASET_NAME,
    "adata_batch_idx": train_batches,              # LUNA train sections only
    "root_data_dir": str(ROOT_DATA_DIR),
    "gene_count_transform_names": ["SubsetHVG"],
    "gene_count_transform_params": {"n_genes": 249},  # full MERFISH panel
    "graph_params": {"spatial_key": "spatial", "delaunay": False, "n_neighs": 10, "radius": None},
    "feature_names": ["X"],
    "label_name": "cell_types",
    "train_transform_names": ["RandomNodeSplit"],
    "train_transform_params": {"val_ratio": 0.1, "test_ratio": 0.0},
  },
  "datamodule": {
    "loader_name": "NeighborLoader",
    "loader_params": {"batch_size": 256},
    "sampler_name": "NeighborSampler",
    "sampler_params": {"num_neighbors": [16,16]},
    "inference_params": {"sample_neighbors_for_inference": False},
  },
  "model": {
    "model_name": "VQNiche",
    "encoder_name": "VQNiche_Encoder",
    "attribute_decoder_name": "MLPSoftmax",
    "adjacency_decoder_name": "MLP_AdjacencyDecoder",
    "predictor_name": "Linear",
    "imputation_params": {
      "mask_strategy": "learnable_parameter",
      "base_mask_ratio": 0.3, "final_mask_ratio": 0.3, "warmup_epochs": 0,
      "deterministic_masking": False, "compute_mask_input_diversity": False, "mask_token_eps": 1e-4,
    },
    "train_metrics_list": ["codebook_utilization","pearson_1hop_nbr"],
    "test_metrics_list": [
      "codebook_utilization","pearson_cell_wise","pearson_1hop_nbr",
      "mmd_1hop_nbr","mmd_pca_1hop_nbr","pearson_gene_wise","pearson_gene_wise_1hop_nbr"
    ],
    "encoder_params": {
      "gnn_name": "SAGEConv",
      "mlp_params": {"hidden_channels": [600,200], "dropout": 0.1, "act": "relu", "norm": None},
      "gnn_params": {               # GNN OFF
        "hidden_channels": 200, "num_layers": 0, "act_first": True,
        "activation": "relu","norm": None,"dropout": 0.0,"init_method": "kaiming_uniform"
      },
      "conditioning_params": {      # FiLM OFF
        "condition_list": None, "use_bias": True, "use_residual": False,
        "residual_weight": 0.2, "init_mode": "identity"
      },
      "vq_params": {                # VQ effectively OFF
        "vq_name": "VectorQuantize",
        "freeze_codebook": False,
        "use_cosine_sim": True, "ema_update": True, "manual_ema_update": False,
        "threshold_ema_dead_code": 100, "manual_in_place_optimizer_update": False,
        "learnable_codebook": False,
        "codebook_size": 512, "heads": 1, "separate_codebook_per_head": False,
        "decay": 0.8, "eps": 1e-5, "kmeans_init": False, "kmeans_iters": 10,
        "sync_kmeans": True, "sample_codebook_temp": 0.0,
        "commitment_weight": 0.0, "commitment_use_cross_entropy_loss": False,
        "codebook_diversity_loss_weight": 0.0, "codebook_diversity_temperature": 100.0,
        "orthogonal_reg_weight": 0.0, "orthogonal_reg_max_codes": None,
        "orthogonal_reg_active_codes_only": False
        # if your branch still *requires* VQ tensors, add: "disabled": True
      },
    },
    "attribute_decoder_params": {   # no conditioning
      "apply_conditioning": None,
      "mlp_params": {"hidden_channels": [600], "dropout": 0.1, "act": "gelu", "norm": "layer_norm"},
    },
    "adjacency_decoder_params": {
      "out_channels": 600,
      "mlp_params": {"hidden_channels": [400], "dropout": 0.2, "act": "relu", "norm": "layer_norm"},
    },
    "optimizer_params": {"optimizer_name": "adam", "lr": 1e-3, "weight_decay": 1e-3, "mask_lr_scale": 2.0},
    "loss_params": {
      "loss_names": ["nb_attribute_reconstruction_loss","bce_adjacency_reconstruction_loss",
                     "mse_commit_loss","mse_code_loss","mask_token_regularization"],
      "loss_kwargs": {
        "k_hop_nb_loss": 1, "only_masked": False, "edge_sampling_ratio": None, "use_pos_weight": True,
        "estimate_adj_kwargs": {"nonlinearity": "sigmoid","k": 8},
        "wt_cross_entropy": 1.0, "wt_attr_reconstr": 1.0, "wt_adj_reconstr": 1.0,
        "wt_joint_code_commit": 0.0, "wt_commit": 0.0, "wt_code": 0.0,
        "wt_mask_token_regularization": 1.0,
      },
    },
  },
  "trainer": {
    "monitor": "val_pearson_1hop_nbr",
    "checkpoint_params": {"mode": "max", "save_top_k": 1, "save_last": True},
    "max_epochs": 10,
    "enable_checkpointing": True,
    "ckpt_path": "best",
  },
}

with open(TRAIN_CFG, "w") as f:
    yaml.safe_dump(yaml_cfg, f, sort_keys=False)

print("Wrote training config to:\n", TRAIN_CFG)


Wrote training config to:
 /nfs/team361/mv11/vqniche-reproducibility/config/train_model/merfish_mouse_cortex_from_luna_csv_luna-split_vniche-mlp.yaml


### Train

In [22]:
import sys
print(sys.executable)  # should be /nfs/team361/mv11/.venvs/LUNA/bin/python
!{sys.executable} -m pip install -q --no-cache-dir scikit-misc==0.3.1


/nfs/team361/mv11/.venvs/LUNA/bin/python


In [26]:
import sys, yaml
sys.path.insert(0, "/nfs/team361/mv11/vqniche/src")

from vqniche.initializers.initialize import (
    initialize_dataset_blob,
    initialize_databatch,
    initialize_datamodule,
    set_model_class,
)

import pytorch_lightning as pl
import torch

TRAIN_CFG = "/nfs/team361/mv11/vqniche-reproducibility/config/train_model/merfish_mouse_cortex_from_luna_csv_luna-split_vniche-mlp.yaml"

pl.seed_everything(42, workers=True)
print("Python:", sys.executable)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())

cfg = yaml.safe_load(open(TRAIN_CFG))

dataset_blob = initialize_dataset_blob(cfg)          # loads existing gold blob (no rebuild)
data_train   = initialize_databatch(cfg, dataset_blob)
datamodule   = initialize_datamodule(cfg, data_train)


Seed set to 42


Python: /nfs/team361/mv11/.venvs/LUNA/bin/python
CUDA available: True | GPUs: 1


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/io/fs.py:229: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "


SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: Subsetted data to 249 features.
SubsetHVG: 

In [27]:
import inspect

ModelClass = set_model_class(cfg["model"]["model_name"])

def build_model(cfg, data_train):
    sig = inspect.signature(ModelClass.__init__)
    params = sig.parameters

    # common variants across branches
    if "config" in params:
        return ModelClass(config=cfg)
    if "cfg" in params:
        return ModelClass(cfg=cfg)
    if "hparams" in params:
        return ModelClass(hparams=cfg)

    # Fallback: pass only args the class actually expects
    # Start with model sub-config
    model_kwargs = dict(cfg["model"])

    # Provide dims if accepted by this class
    if "num_features" in params:
        model_kwargs["num_features"] = int(getattr(data_train, "num_features"))
    if "num_classes" in params:
        model_kwargs["num_classes"] = int(getattr(data_train, "num_classes"))

    # Filter to keys that appear in the constructor
    filtered = {k: v for k, v in model_kwargs.items() if k in params}
    try:
        return ModelClass(**filtered)
    except TypeError as e:
        # Last-resort: try no-arg constructor if it exists
        try:
            return ModelClass()
        except Exception:
            raise e

model = build_model(cfg, data_train)
print("Built model:", type(model).__name__)


TypeError: randn(): argument 'size' (position 1) must be tuple of ints, but found element of type NoneType at pos 0

In [ ]:

ModelClass = set_model_class(cfg["model"]["model_name"])
model = ModelClass(config=cfg)

trainer = pl.Trainer(
    accelerator="auto",
    devices="auto",
    max_epochs=cfg["trainer"]["max_epochs"],
    deterministic=True,
    enable_checkpointing=True,
)

print("Fitting…")
trainer.fit(model=model, datamodule=datamodule)


In [ ]:
from pathlib import Path
import yaml

cfg_test = yaml.safe_load(open(TRAIN_CFG))
cfg_test["dataset"]["adata_batch_idx"] = test_batches

dataset_blob_test = initialize_dataset_blob(cfg_test)
data_test = initialize_databatch(cfg_test, dataset_blob_test)
dm_test = initialize_datamodule(cfg_test, data_test)

Model = set_model_class(cfg_test["model"]["model_name"])

# pick best.ckpt if present
best_ckpt = None
for p in Path(RUN_ROOT).rglob("*.ckpt"):
    if "last.ckpt" not in p.name:
        best_ckpt = str(p); break

best_model = Model.load_from_checkpoint(best_ckpt) if best_ckpt else model
best_model.eval()

trainer.test(model=best_model, datamodule=dm_test,
             ckpt_path=best_ckpt if best_ckpt else None)


In [ ]:
import pickle
infer_loader = dm_test.infer_dataloader()
infer_dict = best_model.collect_inference_data(infer_loader)

with open(Path(dataset_blob.processed_dir) / "label_categories.pkl", "rb") as f:
    infer_dict["label_categories"] = pickle.load(f)

res_dir = (Path(trainer.logger.experiment.dir) if yaml_cfg["logging"]["enabled"] else RUN_ROOT) / "results"
res_dir.mkdir(parents=True, exist_ok=True)
with open(res_dir / "inference_data_dict_merfish_test.pkl", "wb") as f:
    pickle.dump(infer_dict, f)

print("Saved:", res_dir / "inference_data_dict_merfish_test.pkl")


In [44]:
!python /nfs/team361/mv11/vqniche-reproducibility/scripts/train_vanilla_mlp.py \
  --config /nfs/team361/mv11/vqniche-reproducibility/config/train_model/merfish_mouse_cortex_from_luna_csv_luna-split_vniche-mlp.yaml


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: libcudart.so.11.0: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatt

In [41]:
!python /nfs/team361/mv11/vqniche-reproducibility/scripts/train_vanilla_mlp.py \
  --config /nfs/team361/mv11/vqniche-reproducibility/config/train_model/merfish_mouse_cortex_from_luna_csv_vanilla-mlp.yaml

/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: libcudart.so.11.0: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatt